# CHSE Phase 3 — HOE, Welfare Analysis, and Empirical Pipeline

This notebook covers the complete Phase 3 of the CHSE implementation:

1. **HOE via BenchmarkSim** — convergence from diverse initial states (Section 8)
2. **Full Markov chain** — all four mechanisms, stable regime HOE (Section 5)
3. **Ergodicity conditions** — irreducibility and aperiodicity (Prop. on Ergodicity)
4. **Lyapunov stability** — V(s) verification (Theorem 8.2)
5. **Three welfare distortions** — with monetised welfare loss (Bottleneck 6)
6. **Hierarchy Persistence Paradox** — calibrated, ∂E[cascade]/∂HSI > 0 (Bottleneck 8)
7. **Empirical pipeline** — FDI, Figure 5, phase prediction, paradox test (Section 10–11)

**New in Phase 3**: `chse.core.simulation` — unified `BenchmarkSim` / `FullSim` interface.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from chse.core.primitives import Params
from chse.core.network import CHSENetwork
from chse.core.simulation import BenchmarkSim, FullSim   # Phase 3 unified interface
from chse.equilibrium.hoe import check_ergodicity_conditions, stationarity_test
from chse.equilibrium.lyapunov import verify_lyapunov
from chse.equilibrium.markov import run_chain
from chse.welfare.distortions import (
    compute_welfare_distortions, reframing_distortion,
    social_optimal_eta, total_clarity_gap,
)
from chse.welfare.paradox import calibrated_paradox_scan
from chse.empirical.fdi import (
    build_paper_examples, hoe_statistics_from_series,
    predict_regime, persistence_paradox_test,
)

plt.rcParams.update({'figure.dpi':120,'font.size':11,
                     'axes.spines.top':False,'axes.spines.right':False})

def make_params(**kw):
    defaults = dict(
        alpha_R=0.4, alpha_I=0.1, beta_I=0.05, lambda_R=1.0,
        lambda_kappa=1.0, lambda_sigma=0.5, mu_II=0.2,
        K_cap=10.0, M_cap=10.0, delta_kappa=0.2,
        c_mu=0.3, c_kappa=0.3, beta_R=0.15, zeta_II=0.3,
    )
    defaults.update(kw)
    return Params(**defaults)

print('All imports OK')
print(f'Default rho_kappa={Params().rho_kappa}  rho_mu={Params().rho_mu}  '
      f'ratio={Params().rho_kappa/Params().rho_mu:.4f} (irrational → aperiodic)')

## 1. HOE estimation via BenchmarkSim

**Definition 8.1** (HOE): An invariant measure π* on (S, B(S)) such that
π*(B) = ∫_S P(s,B) π*(ds) for all B ∈ B(S).

`BenchmarkSim` runs `TwoPlayerModel.integrate_stochastic` from diverse initial
belief values h₀ ∈ {0.20, 0.45, 0.75, 0.90}, then pools post-burn-in samples.

**Stable regime (HSI=2.1)**: all chains converge to h*≈0.80 regardless of start.
This demonstrates π* = δ_{h*} — the degenerate Stackelberg HOE.

**Oscillatory regime (HSI=1.0)**: chains mix around h*=0.5, producing a
non-degenerate π* with positive variance — the interior HOE.

In [ ]:
# HOE estimation — stable regime
sim_stable = BenchmarkSim(regime='stable', T=300, burn_in=80, n_chains=4)
result_stable = sim_stable.run(seed=42)

# HOE estimation — oscillatory regime
sim_osc = BenchmarkSim(regime='oscillatory', T=300, burn_in=80, n_chains=4)
result_osc = sim_osc.run(seed=42)

print('=== Stable HOE (HSI=2.1) ===')
print(result_stable.hoe_stats.summary())
print(f'  Turnover counts: {result_stable.turnover_counts}')
st = result_stable.stationarity_check()
print(f'  Stationarity: max_diff={st["max_diff"]:.4f}  converged={st["converged"]}')

print()
print('=== Oscillatory HOE (HSI=1.0) ===')
print(result_osc.hoe_stats.summary())
print(f'  Turnover counts: {result_osc.turnover_counts}')
st2 = result_osc.stationarity_check()
print(f'  Stationarity: max_diff={st2["max_diff"]:.4f}  converged={st2["converged"]}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

colours_r = {'stable':'#2a7ab5','oscillatory':'#e8a020'}

# Top row: stable regime
ax_h, ax_dist = axes[0]
for i, h in enumerate(result_stable.h_trajectories):
    ax_h.plot(h, lw=1.0, alpha=0.8, label=f'h₀={sim_stable.h0_spread[i]:.2f}')
ax_h.axhline(0.5, color='gray', ls='--', lw=0.8)
ax_h.axhline(result_stable.hoe_stats.mean_h, color='#c0392b', ls='-', lw=1.5,
             label=f'E[h]={result_stable.hoe_stats.mean_h:.3f}')
ax_h.axvline(80, color='#e8a020', ls=':', lw=1, label='Burn-in end')
ax_h.set_title('Stable HOE (HSI=2.1) — chains from diverse h₀', fontsize=11)
ax_h.set_ylabel('h(t)'); ax_h.set_xlabel('Period t'); ax_h.legend(fontsize=8)

ax_dist.hist(result_stable.pooled_h, bins=40, color='#2a7ab5', alpha=0.8,
             edgecolor='white', lw=0.3)
ax_dist.axvline(result_stable.hoe_stats.mean_h, color='#c0392b', lw=2,
                label=f'E[h]={result_stable.hoe_stats.mean_h:.3f}')
ax_dist.axvline(0.5, color='gray', ls='--', lw=0.8)
ax_dist.set_title('Stable π*(h) — concentrated near h*=0.80', fontsize=11)
ax_dist.set_xlabel('h'); ax_dist.set_ylabel('Frequency'); ax_dist.legend(fontsize=9)

# Bottom row: oscillatory regime
ax_h2, ax_dist2 = axes[1]
for i, h in enumerate(result_osc.h_trajectories):
    ax_h2.plot(h, lw=0.8, alpha=0.75, label=f'h₀={sim_osc.h0_spread[i]:.2f}')
ax_h2.axhline(0.5, color='gray', ls='--', lw=0.8)
ax_h2.axvline(80, color='#e8a020', ls=':', lw=1)
ax_h2.set_title('Oscillatory HOE (HSI=1.0)', fontsize=11)
ax_h2.set_ylabel('h(t)'); ax_h2.set_xlabel('Period t'); ax_h2.legend(fontsize=8)

ax_dist2.hist(result_osc.pooled_h, bins=40, color='#e8a020', alpha=0.8,
              edgecolor='white', lw=0.3)
ax_dist2.axvline(result_osc.hoe_stats.mean_h, color='#c0392b', lw=2,
                 label=f'E[h]={result_osc.hoe_stats.mean_h:.3f}')
ax_dist2.axvline(0.5, color='gray', ls='--', lw=0.8)
ax_dist2.set_title('Oscillatory π*(h) — spread around h*=0.5', fontsize=11)
ax_dist2.set_xlabel('h'); ax_dist2.set_ylabel('Frequency'); ax_dist2.legend(fontsize=9)

plt.suptitle('HOE estimation: stable vs oscillatory regime (Definition 8.1 / Definition 10.2)',
             y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

## 2. Full Markov chain simulation

`FullSim` runs the complete period timeline from Section 5 with all four mechanisms:

- **Mechanism I**: Beta-Binomial anticipation signals
- **Mechanism II**: Role ambiguity (mean reversion toward 0.5)
- **Mechanism III**: Retroactive reframing (optimal best-response)
- **Mechanism IV**: Contagious propagation via kernel K

The full chain naturally demonstrates the **stable HOE** (π* = δ_{h*}) because
the optimal best-response follower only attacks when h < 0.5 (the leader is ahead).
This is correct: for HSI >> 1, the stable fixed point is globally attracting.

In [ ]:
net = CHSENetwork.two_player(h0=0.65)
p = make_params(HSI=2.1)

# Show single chain trajectory first
chain = run_chain(net, p, T=200, seed=42)
print(f'Chain: T={chain.T}, {chain.n_edges} edge, {chain.n_players} players')
print(f'h final: {chain.h_trajectory[-1].round(4)}')
print(f'Turnover frequency: {chain.turnover_frequency():.4f} flips/period')
print(f'h mean (post-50):   {chain.h_mean():.4f}')
print(f'h var  (post-50):   {chain.h_variance():.6f}')
print(f'Regime: stable HOE — h converges to h*≈{0.5+(0.8-0.2*1.0)/(2*2.0):.2f} (fixed point)')

fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
t = np.arange(chain.T+1)
h = chain.h_trajectory[:, 0]
axes[0].plot(t, h, color='#2a7ab5', lw=1.2)
axes[0].axhline(0.5, color='gray', ls='--', lw=0.8)
axes[0].fill_between(t, 0.5, h, where=(h>=0.5), alpha=0.15, color='#2a7ab5')
axes[0].set_ylabel('h(t)'); axes[0].set_title('Markov chain — stable HOE (HSI=2.1)')
axes[1].plot(t, chain.kappa_traj[:,0], color='#2a9d8f', lw=1.2)
axes[1].axhline(p.K_cap/2, color='gray', ls='--', lw=0.8, alpha=0.5)
axes[1].set_ylabel('κ₁(t)')
axes[2].plot(t, chain.mu_traj[:,1], color='#e76f51', lw=1.2)
axes[2].axhline(p.M_cap/2, color='gray', ls='--', lw=0.8, alpha=0.5)
axes[2].set_ylabel('μ₂(t)'); axes[2].set_xlabel('Period t')
plt.tight_layout(); plt.show()

In [ ]:
# FullSim: 4 chains from diverse initial states
fsim = FullSim(net, p, T=200, burn_in=60, n_chains=4)
fsim_result, full_chains = fsim.run(seed=42)

print('=== FullSim HOE Statistics (stable regime) ===')
print(fsim_result.hoe_stats.summary())
st = fsim_result.stationarity_check()
print(f'Stationarity: windows={st["window_means"]}  max_diff={st["max_diff"]:.4f}  converged={st["converged"]}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for i, h in enumerate(fsim_result.h_trajectories):
    ax1.plot(h, lw=0.9, alpha=0.7, label=f'Chain {i+1}')
ax1.axhline(0.5, color='gray', ls='--', lw=0.8)
ax1.axvline(60, color='#e8a020', ls=':', lw=1, label='Burn-in end')
ax1.set_xlabel('Period t'); ax1.set_ylabel('h(t)')
ax1.set_title('4 full Markov chains — all converge to stable HOE')
ax1.legend(fontsize=8)

ax2.hist(fsim_result.pooled_h, bins=30, color='#2a7ab5', alpha=0.8,
         edgecolor='white', lw=0.3)
ax2.axvline(fsim_result.hoe_stats.mean_h, color='#c0392b', lw=2,
            label=f'E[h]={fsim_result.hoe_stats.mean_h:.3f}')
ax2.set_xlabel('h'); ax2.set_ylabel('Frequency')
ax2.set_title('Stable π*(h) from full Markov engine')
ax2.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 3. Ergodicity conditions

**Proposition (Ergodicity)**: If the induced Markov process is:

1. **Irreducible** — every open set B ⊂ S has positive probability of being
   reached from any s in finite steps (sufficient: all λ_R, λ_κ, λ_σ > 0
   and replenishment rates ρ_κ, ρ_μ > 0).
2. **Aperiodic** — sufficient condition: ρ_κ/ρ_μ is irrational.

Then π* is **unique** and ‖P^t(s,·) − π*‖_TV → 0 from any s(0).

*Note*: We set ρ_κ = 0.31, ρ_μ = 0.30 by default so their ratio
(≈1.033) is not a simple rational, guaranteeing aperiodicity.

In [ ]:
p_erg = Params()  # uses default rho_kappa=0.31, rho_mu=0.30
erg = check_ergodicity_conditions(CHSENetwork.two_player(), p_erg)

print('=== Ergodicity Conditions ===')
print(f'Irreducible : {erg["irreducible"]}')
print(f'Aperiodic   : {erg["aperiodic"]}  '
      f'(rho_kappa/rho_mu = {erg["rho_ratio"]:.6f})')
print(f'Ergodic     : {erg["ergodic"]}')
print()
print('Irreducibility checks:')
for k, v in erg['irreducibility_checks'].items():
    print(f'  {"OK  " if v else "FAIL"} {k}')

# Also show what happens with bad params (rho_kappa = rho_mu)
p_bad = Params(rho_kappa=0.5)  # rho_kappa = rho_mu = 0.5 -> ratio=1.0 -> periodic
erg_bad = check_ergodicity_conditions(CHSENetwork.two_player(), p_bad)
print(f'\nWith rho_kappa=0.5 (= rho_mu): aperiodic={erg_bad["aperiodic"]}  '
      f'ergodic={erg_bad["ergodic"]}')

## 4. Lyapunov stability

**Theorem 8.2**: Let π* be an HOE. If Γ < (1−δ)/(1+δ), then π* is Lyapunov stable.

    V(s) = d(s, Ω*)² + Σᵢ[θ_κ(κᵢ−κᵢ*)² + θ_μ(μᵢ−μᵢ*)²]

We verify that ΔV < 0 on average and check the formal condition.

In [ ]:
chain_lyap = run_chain(CHSENetwork.two_player(h0=0.65),
                       make_params(discount=0.9), T=300, seed=42)
result_lyap = verify_lyapunov(chain_lyap, make_params(discount=0.9),
                               burn_in=80, n_support_points=40, Gamma=0.25)
print(result_lyap.summary())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(result_lyap.V_trajectory, color='#2a7ab5', lw=1.2)
ax1.set_xlabel('Period (post burn-in)'); ax1.set_ylabel('V(s)')
ax1.set_title('Lyapunov function V(s) along trajectory')

ax2.hist(result_lyap.delta_V, bins=40, color='#e76f51', alpha=0.8,
         edgecolor='white', lw=0.3)
ax2.axvline(0, color='black', lw=0.8)
ax2.axvline(result_lyap.mean_delta_V, color='#c0392b', lw=2,
            label=f'mean ΔV={result_lyap.mean_delta_V:.6f}')
ax2.set_xlabel('ΔV per period'); ax2.set_ylabel('Frequency')
ax2.set_title(f'ΔV distribution ({result_lyap.frac_decreasing:.1%} steps ΔV<0)')
ax2.legend(fontsize=9)
plt.tight_layout(); plt.show()

## 5. Three welfare distortions

In any interior HOE π*, relative to the social optimum (Bottleneck 6):

| Distortion | Direction | Magnitude | Policy fix |
|-----------|-----------|-----------|------------|
| 1. Reframing | Over-investment | ∝ β_R·Γ/(1−Γ) | Legal estoppel, institutional precedent |
| 2. Commitment resistance | Over-investment | ∝ Σ_k φ(d(i,k))·u_k^coord | Legibility subsidies |
| 3. Hierarchy clarity | Under-investment | ∝ ζ_II·\|E_i\| | Public commitment requirements |

**Welfare loss** = monetised sum of all three distortions.

In [ ]:
net3 = CHSENetwork.complete(3, initial_h=0.65)
p_w = make_params(beta_R=0.15, zeta_II=0.3)

wd = compute_welfare_distortions(
    net3, p_w, eta_eq=0.5, kappa_eq=0.6,
    u_L=10.0, u_F=2.0, Gamma=0.4, u_coordination=0.5
)
print(wd.summary())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Distortion 1: reframing vs Gamma
G_vals = np.linspace(0.0, 0.85, 100)
excess = [reframing_distortion(net3, p_w, 0.5, g) for g in G_vals]
eta_so = [social_optimal_eta(net3, p_w, 0.5, g) for g in G_vals]
axes[0].plot(G_vals, excess, color='#c0392b', lw=2, label='Excess η (over-investment)')
axes[0].plot(G_vals, [0.5]*len(G_vals), color='gray', ls='--', lw=1, label='η_eq=0.5')
axes[0].plot(G_vals, eta_so, color='#2a7ab5', lw=2, label='η_SO (social optimum)')
axes[0].set_xlabel('Propagation factor Γ')
axes[0].set_ylabel('Reframing investment')
axes[0].set_title('Distortion 1: over-investment in reframing')
axes[0].legend(fontsize=8)

# Distortion 3: clarity gap vs HSI (lower HSI = more contested = more ambiguity)
hsi_w = [0.4, 0.7, 1.0, 1.5, 2.1, 2.5, 3.0]
gaps = []
for hsi in hsi_w:
    h_val = min(0.95, 0.5 + 0.4/hsi)
    nt = CHSENetwork.complete(3, initial_h=h_val)
    gaps.append(total_clarity_gap(nt, p_w))
axes[1].bar(range(len(hsi_w)), gaps, color='#e8a020', alpha=0.85)
axes[1].set_xticks(range(len(hsi_w)))
axes[1].set_xticklabels([str(h) for h in hsi_w], fontsize=9)
axes[1].set_xlabel('HSI'); axes[1].set_ylabel('Clarity gap')
axes[1].set_title('Distortion 3: under-investment in hierarchy clarity\n(higher contest → more ambiguity)')

# Welfare loss breakdown
labels = ['Reframing\n(D1)', 'Resistance\n(D2)', 'Clarity\n(D3)', 'Total']
d1 = wd.reframing_excess * p_w.c_mu * len(net3.canon_edges)
d2 = wd.resistance_excess * p_w.c_kappa * len(net3.canon_edges)
d3 = wd.clarity_gap
values = [d1, d2, d3, wd.welfare_loss]
bar_colors = ['#e76f51', '#e8a020', '#c0392b', '#2a7ab5']
axes[2].bar(labels, values, color=bar_colors, alpha=0.85)
axes[2].set_ylabel('Welfare loss (monetised)')
axes[2].set_title('Welfare loss breakdown\n(D1+D2+D3 = Total)')

plt.tight_layout(); plt.show()
print(f'Total welfare loss: {wd.welfare_loss:.4f}')

## 6. Hierarchy Persistence Paradox

**Theorem (Bottleneck 8)**: Conditional on a leadership collapse, the expected
cascade size is *increasing* in HSI.

Causal chain:
1. ∂Acc_ij/∂HSI > 0 — stronger leaders predict better
2. ∂ρ(K)/∂Acc_ij > 0 — higher accuracy inflates K weights
3. ∂E[cascade]/∂ρ(K) = α_R/(1−ρ(K))² > 0

**Combined**: ∂E[cascade|collapse]/∂HSI > 0

Calibration: Acc_ij(HSI) = 0.50 + 0.42·HSI/(1+HSI), giving ρ(K) ∈ [0.23, 0.32]
with trust=0.65 and φ=0.60 (moderate network connectivity).

In [ ]:
pr = calibrated_paradox_scan(
    hsi_vals=np.linspace(0.3, 3.5, 300),
    alpha_R=0.5, trust_avg=0.65, phi_avg=0.60,
)
print(pr.summary())

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(pr.hsi_vals, pr.acc_vals, color='#2a7ab5', lw=2)
axes[0].set_xlabel('HSI'); axes[0].set_ylabel('Acc_ij')
axes[0].set_title('Anticipation accuracy\nincreases with HSI')

axes[1].plot(pr.hsi_vals, pr.rho_K_vals, color='#e8a020', lw=2)
axes[1].axhline(1.0, color='gray', ls='--', lw=0.8, label='ρ(K)=1')
axes[1].set_xlabel('HSI'); axes[1].set_ylabel('ρ(K)')
axes[1].set_title('ρ(K) inflates with HSI\n(via anticipation accuracy)')
axes[1].legend(fontsize=9)

valid = ~np.isnan(pr.cascade_sizes)
axes[2].plot(pr.hsi_vals[valid], pr.cascade_sizes[valid], color='#c0392b', lw=2)
axes[2].set_xlabel('HSI'); axes[2].set_ylabel('E[cascade | collapse]')
axes[2].set_title('Hierarchy Persistence Paradox:\nstronger leaders → larger collapse cascade')

plt.suptitle('∂E[cascade|collapse]/∂HSI > 0', y=1.02, fontsize=12)
plt.tight_layout(); plt.show()
print(f'Derivative positive: {pr.derivative_sign}')
print(f'Cascade range: [{np.nanmin(pr.cascade_sizes):.4f}, {np.nanmax(pr.cascade_sizes):.4f}]')

## 7. Empirical pipeline (Section 10–11)

### Step 3 — Fiscal Dominance Index (Corollary 11.1)

    FDI = V_T · λ_R / (K_CB · ρ_κ/ρ_ν)

| FDI | Regime |
|-----|--------|
| < 0.5 | Monetary dominance |
| 0.5–1.0 | Contested |
| > 1.0 | Fiscal dominance |

In [ ]:
examples = build_paper_examples()

print('=== Figure 5 — Estimated Fiscal Dominance Index ===')
print(f'{"Country":<12} {"Period":<10} {"V_T":<6} {"K_CB":<6} {"FDI":<8} {"Regime"}')
print('-' * 55)
for e in examples:
    print(f'{e.country:<12} {e.period:<10} {e.V_T:<6.2f} {e.K_CB:<6.2f} {e.FDI:<8.2f} {e.regime}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colours_fdi = {'monetary':'#2a7ab5','contested':'#e8a020','fiscal':'#c0392b'}
labels = [f'{e.country}\n({e.period})' for e in examples]
fdis = [e.FDI for e in examples]
bar_cols = [colours_fdi[e.regime] for e in examples]
bars = ax.barh(labels, fdis, color=bar_cols, alpha=0.85, edgecolor='white', lw=0.5)
ax.axvline(0.5, color='gray', ls='--', lw=1, label='FDI=0.5 (borderline)')
ax.axvline(1.0, color='black', ls='--', lw=1.5, label='FDI=1 (fiscal threshold)')
for bar, fdi in zip(bars, fdis):
    ax.text(fdi+0.02, bar.get_y()+bar.get_height()/2,
            f'{fdi:.2f}', va='center', fontsize=9)
patches = [mpatches.Patch(color=v, label=k.capitalize())
           for k,v in colours_fdi.items()]
ax.legend(handles=patches+[ax.lines[0],ax.lines[1]], fontsize=9, loc='lower right')
ax.set_xlabel('Fiscal Dominance Index (FDI = V_T·λ_R / (K_CB·ρ_κ/ρ_ν))')
ax.set_title('Figure 5 — Estimated FDI for selected economies')
plt.tight_layout(); plt.show()

### Step 4 — Phase diagram prediction test

Verify that the FDI-based regime classification is consistent with CHSE theory.

In [ ]:
print('=== Step 4: Phase diagram prediction ===')
print(f'{"Country":<12} {"FDI":<6} {"HSI":<6} {"Z":<7} {"Predicted":<11} {"FDI regime":<12} {"OK?"}')
print('-'*65)
all_ok = True
for ex in examples:
    pred = predict_regime(ex, pi=0.0)
    ok = 'YES' if pred['consistent'] else 'NO'
    if not pred['consistent']: all_ok = False
    print(f'{ex.country:<12} {pred["FDI"]:<6.2f} {pred["HSI"]:<6.2f} '
          f'{pred["Z"]:<7.4f} {pred["predicted_regime"]:<11} '
          f'{pred["fdi_regime"]:<12} {ok}')
print(f'\nAll consistent: {all_ok}')

### Step 5 — Hierarchy Persistence Paradox test

Regress post-collapse yield volatility on pre-collapse HSI. Positive coefficient predicted.

In [ ]:
ptest = persistence_paradox_test(examples)
print('=== Step 5: Paradox test ===')
print(f'Correlation HSI ~ post-collapse volatility: {ptest["correlation"]}')
print(f'Paradox confirmed (r>0): {ptest["paradox_confirmed"]}')
print(f'Note: volatility is simulated for illustration ({"simulated" if ptest["simulated"] else "observed"})')

fig, ax = plt.subplots(figsize=(7, 4))
finite = [(h,v,e.country) for h,v,e in
          zip(ptest['hsi_vals'], ptest['volatility'], examples) if h<1e5]
hsi_f,vol_f,cnames = zip(*finite) if finite else ([],[],[])
ax.scatter(hsi_f, vol_f, color='#c0392b', s=90, zorder=5)
if len(hsi_f)>=2:
    z=np.polyfit(hsi_f,vol_f,1)
    xl=np.linspace(min(hsi_f),max(hsi_f),100)
    ax.plot(xl,np.polyval(z,xl),color='#c0392b',ls='--',lw=1.5,
            label=f'slope={z[0]:.3f}')
for h,v,n in zip(hsi_f,vol_f,cnames):
    ax.annotate(n,(h,v),xytext=(h+0.05,v+0.01),fontsize=8)
ax.set_xlabel('HSI (pre-collapse)')
ax.set_ylabel('Post-collapse yield volatility')
ax.set_title('Paradox test: stronger hierarchies → larger collapse disruption')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

### HOE statistics from observed h(t) series

Definition 10.2: HOE ≡ π*(τ̂, Var(h), E[cascade size])

In [ ]:
# Simulate a realistic observed h(t) series using the oscillatory benchmark
from chse.benchmark.two_player import TwoPlayerModel
model_obs = TwoPlayerModel(mu=0.6, eta_bar=0.4, kappa_bar=0.4, r_bar=1.0,
                           h0=0.5, params=Params(HSI=1.0))
obs_result = model_obs.integrate_stochastic(T=500, noise_std=0.09, seed=42)
hoe_obs = hoe_statistics_from_series(obs_result.h)
print('=== HOE statistics from observed h(t) (oscillatory regime) ===')
print(hoe_obs.summary())

## Summary

| Component | Paper reference | Status |
|-----------|----------------|--------|
| BenchmarkSim: stable HOE convergence | Theorem 8.1, Def 10.2 | OK |
| BenchmarkSim: oscillatory HOE distribution | Theorem 8.1 | OK |
| FullSim: 4-mechanism Markov chain | Section 5, Theorem 8.1 | OK |
| Ergodicity: irreducibility checks | Proposition on Ergodicity | OK |
| Ergodicity: aperiodicity (rho_k/rho_m irrational) | Proposition on Ergodicity | OK |
| Lyapunov V(s), ΔV distribution | Theorem 8.2 | OK |
| Welfare distortion 1: reframing | Bottleneck 6 | OK |
| Welfare distortion 2: resistance | Bottleneck 6 | OK |
| Welfare distortion 3: clarity | Bottleneck 6 | OK |
| Welfare loss (monetised) | Bottleneck 6 | OK |
| Hierarchy Persistence Paradox | Bottleneck 8 | OK |
| FDI Figure 5 reproduction | Corollary 11.1 | OK |
| Phase diagram prediction test | Theorem 6.1–6.2 | OK |
| Paradox test (positive correlation) | Bottleneck 8 | OK |
| HOE statistics from observed data | Definition 10.2 | OK |